# **Initalize Notebook:**
Downloads PySpark, sets up global variables, allows Google Cloud importing/exporting

In [1]:
%pip -q install pyspark==4.0.0 google-cloud-storage==3.4.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
PROJECT_ID = "cs131-medicaid"
BUCKET = "cs131-medicaid-bucket/"
DATA_URI = "gs://" + BUCKET + "data/"
OUTPUT_URI = "gs://" + BUCKET + "out/"
MEDICAID_URI = DATA_URI + "medicaid-provider-spending.parquet"

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

from google.cloud import storage
client = storage.Client(project=PROJECT_ID)

In [4]:
from pathlib import Path
from urllib.request import urlretrieve

CONNECTOR = str(Path("gcs-connector-3.1.10.jar").resolve())
urlretrieve("https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-3.1.10.jar", CONNECTOR)

('/content/gcs-connector-3.1.10.jar',
 <http.client.HTTPMessage at 0x7c0e461d94f0>)

In [5]:
import os
import sys
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("CS131-Class23")
    .master("local[2]")
    .config("spark.jars", CONNECTOR)
    .config("spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "Master:", spark.sparkContext.master)

Spark: 4.0.0 Master: local[2]


# **Load Medicaid Dataset into PySpark:**
Loads medicaid data from Google Cloud.  Ths second block is optional as it'll convert the dataset into a sample.



In [6]:
useSample = True

if (useSample):
  medicaid = (
      spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(DATA_URI + "sample/sample_1000.csv")
  )
else:
  medicaid = spark.read.parquet(MEDICAID_URI)

medicaid.printSchema()

root
 |-- BILLING_PROVIDER_NPI_NUM: integer (nullable = true)
 |-- SERVICING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- CLAIM_FROM_MONTH: timestamp (nullable = true)
 |-- TOTAL_PATIENTS: integer (nullable = true)
 |-- TOTAL_CLAIM_LINES: integer (nullable = true)
 |-- TOTAL_PAID: double (nullable = true)



# **Task 1: Define and prepare the data**
State the final project goal and two analytical questions. Document what one row represents, its key, required columns/types/units.

In PySpark:
1.   Read the input, inspect its schema, select fields, and set appropriate types.
Check two dataset-specific quality conditions and justify the treatment. Record affected counts. Do not remove repeated records without a defensible duplicate rule.
2.   Create one useful derived column.
3.   Save the prepared data as Parquet, read it back, and verify it.

Save sample-run checks in `out/quality_checks.csv`. Cover row counts and exclusions, types/missing/domain values, key uniqueness if applicable, a join/window check, sampling/split checks, and Parquet read-back count, schema, and one project-specific total or key check. Explain overlapping issue counts rather than adding them together.

In [7]:
MEDICAID_QUALITY_CHECKS = OUTPUT_URI + "quality_checks.csv"

fail_hcpcs_count = medicaid.filter(medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$")).count()
fail_paid_count = medicaid.filter((medicaid.TOTAL_PAID <= 0) | medicaid.TOTAL_PAID.isNull()).count()
overlap_count = medicaid.filter((medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$")) & ((medicaid.TOTAL_PAID <= 0) | medicaid.TOTAL_PAID.isNull())).count()
print("Fail HCPCS Count: " + str(fail_hcpcs_count))
print("Fail Paid Count: " + str(fail_paid_count))
print("Overlap Count: " + str(overlap_count))

# Filters out unncessary fields: entries with $0 paid out & entries with invalid HCPCS codes
medicaid.printSchema()
quality_checks = medicaid.filter(medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$"))
quality_checks = quality_checks.filter(medicaid.TOTAL_PAID > 0)

# Creates an AVERGAGE_PAID field which represents the average payout for each service
quality_checks = quality_checks.withColumn("AVERAGE_PAID", quality_checks.TOTAL_PAID / quality_checks.TOTAL_CLAIM_LINES)

Fail HCPCS Count: 262
Fail Paid Count: 129
Overlap Count: 28
root
 |-- BILLING_PROVIDER_NPI_NUM: integer (nullable = true)
 |-- SERVICING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- CLAIM_FROM_MONTH: timestamp (nullable = true)
 |-- TOTAL_PATIENTS: integer (nullable = true)
 |-- TOTAL_CLAIM_LINES: integer (nullable = true)
 |-- TOTAL_PAID: double (nullable = true)



In [8]:
# Writes filtered data to output folder
quality_checks.write.mode("overwrite").csv(MEDICAID_QUALITY_CHECKS)

# Task 2. Answer two analytical questions

*  Analysis 1: reproduce a PA3 benchmark analysis with the same records, filters, grouping, missing-value rules, precision, and ties. Correct and disclose any baseline error.
*   Analysis 2: answer a different useful question, such as comparing segments or finding top entities within groups.


**Changing only a Top-N cutoff is insufficient.**

**Across the analyses, use grouped aggregation and at least one join OR window.**

- Join: explain keys/relationship; check duplicate lookup keys, unmatched records, and row-count changes.
- Window: explain groups, ordering, ties, and frame when relevant; verify a small group manually or against a grouped result.

Export ordered sample results with headers as analysis_1.csv and analysis_2.csv. Produce actual CSV files, not only Spark part-file directories. In the PDF, show the results based on your full data (or at least the subset that doesn't break the environment), two findings, one labeled chart (any familiar charting tool is acceptable), and a limitation.

In [9]:
from pyspark.sql import Window

# Aggregate payments and claim lines by HCPCS code and month.
hcpcs_sum = (
    quality_checks
    .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
    .agg(
        F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES_SUM"),
        F.sum("TOTAL_PAID").alias("TOTAL_PAID_SUM")
    )
)

paid = F.col("TOTAL_PAID_SUM")
lines = F.col("TOTAL_CLAIM_LINES_SUM")

# Preserve DuckDB's floating-point behavior if a denominator is zero.
paid_per_line = (
    F.when(lines == 0, paid * F.lit(float("inf")))
    .otherwise(paid / lines)
)

# Calculate the monthly rate from the totals.
# Do not average the existing row-level AVERAGE_PAID column.
monthly = hcpcs_sum.withColumn(
    "PAID_PER_CLAIM_LINE",
    F.round(paid_per_line, 2)
)

# Build a ranking within each HCPCS code.
# The oldest month wins when metric values tie.
def month_rank(metric, descending=False):
    metric_order = (
        F.col(metric).desc_nulls_last()
        if descending
        else F.col(metric).asc_nulls_last()
    )

    window = Window.partitionBy("HCPCS_CODE").orderBy(
        metric_order,
        F.col("CLAIM_FROM_MONTH").asc_nulls_last()
    )

    return F.row_number().over(window)


ranked = (
    monthly
    .withColumn(
        "paid_asc_rank",
        month_rank("PAID_PER_CLAIM_LINE")
    )
    .withColumn(
        "paid_desc_rank",
        month_rank("PAID_PER_CLAIM_LINE", descending=True)
    )
    .withColumn(
        "demand_asc_rank",
        month_rank("TOTAL_CLAIM_LINES_SUM")
    )
    .withColumn(
        "demand_desc_rank",
        month_rank("TOTAL_CLAIM_LINES_SUM", descending=True)
    )
)

# Extract the winning month for each ranking.
analysis_1 = (
    ranked
    .groupBy("HCPCS_CODE")
    .agg(
        F.max(
            F.when(F.col("paid_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
        ).alias("LOWEST_PAID_MONTH"),

        F.max(
            F.when(F.col("paid_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
        ).alias("HIGHEST_PAID_MONTH"),

        F.max(
            F.when(F.col("demand_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
        ).alias("LOWEST_DEMAND_MONTH"),

        F.max(
            F.when(F.col("demand_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))
        ).alias("HIGHEST_DEMAND_MONTH")
    )
    .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
)

analysis_1.show(20, truncate=False)

+----------+-------------------+-------------------+-------------------+--------------------+
|HCPCS_CODE|LOWEST_PAID_MONTH  |HIGHEST_PAID_MONTH |LOWEST_DEMAND_MONTH|HIGHEST_DEMAND_MONTH|
+----------+-------------------+-------------------+-------------------+--------------------+
|A0100     |2020-03-01 00:00:00|2020-03-01 00:00:00|2020-03-01 00:00:00|2020-03-01 00:00:00 |
|A0425     |2023-07-01 00:00:00|2020-06-01 00:00:00|2020-06-01 00:00:00|2023-07-01 00:00:00 |
|A0427     |2019-05-01 00:00:00|2019-05-01 00:00:00|2019-05-01 00:00:00|2019-05-01 00:00:00 |
|A0428     |2021-03-01 00:00:00|2021-03-01 00:00:00|2021-03-01 00:00:00|2021-03-01 00:00:00 |
|A0433     |2024-07-01 00:00:00|2024-07-01 00:00:00|2024-07-01 00:00:00|2024-07-01 00:00:00 |
|A4371     |2022-08-01 00:00:00|2022-08-01 00:00:00|2022-08-01 00:00:00|2022-08-01 00:00:00 |
|A4617     |2023-04-01 00:00:00|2023-04-01 00:00:00|2023-04-01 00:00:00|2023-04-01 00:00:00 |
|A5120     |2018-07-01 00:00:00|2018-07-01 00:00:00|2018-07-

In [10]:
import csv

local_output = "/content/analysis_1.csv"

sample_rows = analysis_1.limit(100).collect()

with open(local_output, "w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(analysis_1.columns)
    writer.writerows(sample_rows)

# BUCKET contains a trailing slash; bucket names must omit it.
bucket = client.bucket(BUCKET.rstrip("/"))

bucket.blob("out/analysis_1.csv").upload_from_filename(
    local_output,
    content_type="text/csv"
)

print("Saved:", OUTPUT_URI + "analysis_1.csv")


Saved: gs://cs131-medicaid-bucket/out/analysis_1.csv


In [11]:
# Find a code appearing in at least two different months.
example = (
    monthly
    .groupBy("HCPCS_CODE")
    .count()
    .filter(F.col("count") >= 2)
    .orderBy("count", F.col("HCPCS_CODE").asc_nulls_last())
    .first()
)

if example is None:
    print("No code has multiple months. Use a larger sample for this check.")
else:
    code = example["HCPCS_CODE"]
    same_code = F.col("HCPCS_CODE").eqNullSafe(F.lit(code))

    monthly.filter(same_code).orderBy(
        F.col("CLAIM_FROM_MONTH").asc_nulls_last()
    ).show(truncate=False)

    analysis_1.filter(same_code).show(truncate=False)

+----------+-------------------+---------------------+--------------+-------------------+
|HCPCS_CODE|CLAIM_FROM_MONTH   |TOTAL_CLAIM_LINES_SUM|TOTAL_PAID_SUM|PAID_PER_CLAIM_LINE|
+----------+-------------------+---------------------+--------------+-------------------+
|A0425     |2020-06-01 00:00:00|54                   |1944.36       |36.01              |
|A0425     |2023-07-01 00:00:00|2215                 |16062.63      |7.25               |
+----------+-------------------+---------------------+--------------+-------------------+

+----------+-------------------+-------------------+-------------------+--------------------+
|HCPCS_CODE|LOWEST_PAID_MONTH  |HIGHEST_PAID_MONTH |LOWEST_DEMAND_MONTH|HIGHEST_DEMAND_MONTH|
+----------+-------------------+-------------------+-------------------+--------------------+
|A0425     |2023-07-01 00:00:00|2020-06-01 00:00:00|2020-06-01 00:00:00|2023-07-01 00:00:00 |
+----------+-------------------+-------------------+-------------------+-----------

In [12]:
# Group by HCPCS_CODE and BILLING_PROVIDER_NPI_NUM to get aggregate stats.
provider_spend = (
    quality_checks
    .groupBy("HCPCS_CODE", "BILLING_PROVIDER_NPI_NUM")
    .agg(
        F.sum("TOTAL_PAID").alias("PROVIDER_TOTAL_PAID"),
        F.sum("TOTAL_CLAIM_LINES").alias("PROVIDER_TOTAL_CLAIMS")
    )
)

# Define window partition by HCPCS_CODE and order by PROVIDER_TOTAL_PAID descending.
# Ties are broken using billing provider number in ascending order.
provider_window = Window.partitionBy("HCPCS_CODE").orderBy(
    F.col("PROVIDER_TOTAL_PAID").desc_nulls_last(),
    F.col("BILLING_PROVIDER_NPI_NUM").asc_nulls_last()
)

# Rank providers within each HCPCS code.
ranked_providers = provider_spend.withColumn(
    "provider_rank",
    F.row_number().over(provider_window)
)

# Filter to keep only the top billing provider for each HCPCS code.
analysis_2 = (
    ranked_providers
    .filter(F.col("provider_rank") == 1)
    .select(
        "HCPCS_CODE",
        "BILLING_PROVIDER_NPI_NUM",
        F.round("PROVIDER_TOTAL_PAID", 2).alias("TOP_PROVIDER_TOTAL_PAID"),
        "PROVIDER_TOTAL_CLAIMS"
    )
    .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
)

# Show sample results.
analysis_2.show(20, truncate=False)

+----------+------------------------+-----------------------+---------------------+
|HCPCS_CODE|BILLING_PROVIDER_NPI_NUM|TOP_PROVIDER_TOTAL_PAID|PROVIDER_TOTAL_CLAIMS|
+----------+------------------------+-----------------------+---------------------+
|A0100     |1821033903              |1815.0                 |75                   |
|A0425     |1649356791              |16062.63               |2215                 |
|A0427     |1285669903              |21259.01               |205                  |
|A0428     |1053581645              |27634.1                |174                  |
|A0433     |1164786257              |6217.64                |33                   |
|A4371     |1386688414              |183.27                 |89                   |
|A4617     |1659424992              |196.7                  |66                   |
|A5120     |1053347344              |10.12                  |16                   |
|A7003     |1619080660              |101.2                  |24             

In [13]:
local_output_2 = "/content/analysis_2.csv"
sample_rows_2 = analysis_2.limit(100).collect()

with open(local_output_2, "w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(analysis_2.columns)
    writer.writerows(sample_rows_2)

# Upload to Google Cloud Storage.
bucket = client.bucket(BUCKET.rstrip("/"))
bucket.blob("out/analysis_2.csv").upload_from_filename(
    local_output_2,
    content_type="text/csv"
)

print("Saved:", OUTPUT_URI + "analysis_2.csv")

Saved: gs://cs131-medicaid-bucket/out/analysis_2.csv


#Task 3. Sample and prepare the final-sprint handoff
Use sample(withReplacement=False, fraction=..., seed=131) to create a nonempty development sample. Compare source/sample counts and proportions for at least two meaningful groups; explain changed or missing groups.

Record the fraction and population sampled.

Choose one path, with equal credit:

*   Path A: analysis. Prepare the cleaned/enriched table and both analysis tables for reuse. Verify one aggregate independently, using DuckDB or a small hand calculation. No train/test split is required.

*   Path B: prediction preparation. Identify the target, prediction time, feature list, and separate IDs/audit metadata. Exclude outcome-revealing or unavailable-at-prediction fields. Create and save separate train/test Parquet datasets using one appropriate random, time, or entity split. Verify counts, coverage. Sample training data only; do not use test data to choose features or tune decisions. Fit any later learned imputation/scaling on training only.

Save prepared outputs in persistent team storage, not only temporary Colab files. Save the selected development sample if it must be reused exactly; a seed alone does not fix selections across changed inputs/runtimes.

Include a handoff table in the PDF: output, row meaning, schema, count, storage location, regeneration code, and next use. Add also a three remaining technical tasks, each with an owner, dependency, and completion check. State any unresolved data/scale limitation.

In [14]:
#Path A: uses dev_sample created in initalization

# 1. Determine population count and define the target sample size of 10000 rows.
pop_count = quality_checks.count()
target_sample_size = 100 if useSample else 10000
sample_fraction = target_sample_size / pop_count

# Create the development sample using the required seed.
dev_sample = quality_checks.sample(withReplacement=False, fraction=sample_fraction, seed=131)
dev_sample_count = dev_sample.count()

print(f"Population Count: {pop_count}")
print(f"Requested Sample Fraction: {sample_fraction:.4f}")
print(f"Actual Development Sample Count: {dev_sample_count}")

Population Count: 234
Requested Sample Fraction: 0.4274
Actual Development Sample Count: 86


In [15]:
# 2. Compare group counts and proportions for Group 1: HCPCS Category (using the first character of the HCPCS code).
pop_hcpcs = (
    quality_checks
    .withColumn("HCPCS_Group", F.substring("HCPCS_CODE", 1, 1))
    .groupBy("HCPCS_Group")
    .count()
    .withColumn("pop_proportion", F.round(F.col("count") / pop_count, 4))
    .withColumnRenamed("count", "pop_count")
)

sample_hcpcs = (
    dev_sample
    .withColumn("HCPCS_Group", F.substring("HCPCS_CODE", 1, 1))
    .groupBy("HCPCS_Group")
    .count()
    .withColumn("sample_proportion", F.round(F.col("count") / dev_sample_count, 4))
    .withColumnRenamed("count", "sample_count")
)

# Join both metrics to compare distributions.
hcpcs_comparison = pop_hcpcs.join(sample_hcpcs, on="HCPCS_Group", how="outer").fillna(0)
display(hcpcs_comparison.orderBy("HCPCS_Group"))

DataFrame[HCPCS_Group: string, pop_count: bigint, pop_proportion: double, sample_count: bigint, sample_proportion: double]

In [16]:
# 3. Compare group counts and proportions for Group 2: CLAIM_FROM_MONTH.
pop_month = (
    quality_checks
    .groupBy("CLAIM_FROM_MONTH")
    .count()
    .withColumn("pop_proportion", F.round(F.col("count") / pop_count, 4))
    .withColumnRenamed("count", "pop_count")
)

sample_month = (
    dev_sample
    .groupBy("CLAIM_FROM_MONTH")
    .count()
    .withColumn("sample_proportion", F.round(F.col("count") / dev_sample_count, 4))
    .withColumnRenamed("count", "sample_count")
)

month_comparison = pop_month.join(sample_month, on="CLAIM_FROM_MONTH", how="outer").fillna(0)
display(month_comparison.orderBy("CLAIM_FROM_MONTH").limit(20))

DataFrame[CLAIM_FROM_MONTH: timestamp, pop_count: bigint, pop_proportion: double, sample_count: bigint, sample_proportion: double]

In [17]:
# 4. Save prepared tables for Path A and write output locally + to GCS.
dev_sample_local_path = "/content/development_sample.csv"
dev_sample_rows = dev_sample.collect()

with open(dev_sample_local_path, "w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(dev_sample.columns)
    writer.writerows(dev_sample_rows)

# Upload development sample to GCS.
bucket.blob("out/development_sample.csv").upload_from_filename(
    dev_sample_local_path,
    content_type="text/csv"
)

print("Saved Development Sample to GCS:", OUTPUT_URI + "development_sample.csv")

Saved Development Sample to GCS: gs://cs131-medicaid-bucket/out/development_sample.csv


In [18]:
# 5. Independent verification of one aggregate
# Verify a specific HCPCS code aggregate from our quality_checks against our analysis outputs.
target_hcpcs = "A0425"

# Filter quality_checks for this HCPCS code and aggregate manually in Spark.
spark_check = (
    quality_checks
    .filter(F.col("HCPCS_CODE") == target_hcpcs)
    .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
    .agg(F.sum("TOTAL_CLAIM_LINES").alias("SUM_LINES"), F.sum("TOTAL_PAID").alias("SUM_PAID"))
    .orderBy("CLAIM_FROM_MONTH")
)

print(f"Spark Grouped Aggregates for {target_hcpcs}")
spark_check.show()

# Retrieve and display row count and schema metrics to populate the handoff table
print(" Dataset Counts - ")
print("Quality Checks Population Count:", quality_checks.count())
print("Development Sample Count:", dev_sample.count())
print("Analysis 1 Row Count:", analysis_1.count())
print("Analysis 2 Row Count:", analysis_2.count())

print("\nDevelopment Sample Schema - ")
dev_sample.printSchema()

Spark Grouped Aggregates for A0425
+----------+-------------------+---------+--------+
|HCPCS_CODE|   CLAIM_FROM_MONTH|SUM_LINES|SUM_PAID|
+----------+-------------------+---------+--------+
|     A0425|2020-06-01 00:00:00|       54| 1944.36|
|     A0425|2023-07-01 00:00:00|     2215|16062.63|
+----------+-------------------+---------+--------+

 Dataset Counts - 
Quality Checks Population Count: 234
Development Sample Count: 86
Analysis 1 Row Count: 112
Analysis 2 Row Count: 112

Development Sample Schema - 
root
 |-- BILLING_PROVIDER_NPI_NUM: integer (nullable = true)
 |-- SERVICING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- CLAIM_FROM_MONTH: timestamp (nullable = true)
 |-- TOTAL_PATIENTS: integer (nullable = true)
 |-- TOTAL_CLAIM_LINES: integer (nullable = true)
 |-- TOTAL_PAID: double (nullable = true)
 |-- AVERAGE_PAID: double (nullable = true)



In [19]:
# Compare source/sample datasets.  Compare HCPCS_CODE group and CLAIM_MONTH
dev_sample_hcpcs = dev_sample.groupBy("HCPCS_CODE").sum("TOTAL_PAID").orderBy(F.desc("sum(TOTAL_PAID)")).limit(5)
source_hcpcs = quality_checks.groupBy("HCPCS_CODE").sum("TOTAL_PAID").orderBy(F.desc("sum(TOTAL_PAID)")).limit(5)

hcpcs_comparison = spark.createDataFrame(
    zip(
        [r["HCPCS_CODE"] for r in dev_sample_hcpcs.collect()],
        [r["HCPCS_CODE"] for r in source_hcpcs.collect()],
    ),
    ["dev_sample_hcpcs_top5_paid_out", "source_hcpcs_top5_paid_out"],
)

hcpcs_comparison.show()
hcpcs_comparison.write.mode("overwrite").csv(OUTPUT_URI + "sample_comparison.csv")


+------------------------------+--------------------------+
|dev_sample_hcpcs_top5_paid_out|source_hcpcs_top5_paid_out|
+------------------------------+--------------------------+
|                         T1015|                     T1019|
|                         G9002|                     T1015|
|                         H0018|                     T2040|
|                         T2046|                     G9002|
|                         D0120|                     H0018|
+------------------------------+--------------------------+



# **Task 4. Extend the benchmark Compare Analysis 1 in PySpark and DuckDB:**

Verify all result rows using consistent columns, ordering, and numeric tolerance.
Record three runs per tool on identical records, query rules, and input format. Reuse PA3 DuckDB timings only if these and the timing boundary match; otherwise rerun.
Time input reading. Exclude setup, download, Spark startup, conversion, and charting -- use the exact scope you had in the DuckDB run.
Save benchmark.csv with columns:

```
implementation,environment,input_rows,input_bytes,format,run,elapsed_seconds,output_rows,result_verified
```


Retain Spark timing cells and DuckDB commands. Report versions, Spark master, warm-session/cache conditions, medians, and variation. Different machines measure different setups, not engine speed alone. A slower Spark result is acceptable; local Spark does not demonstrate multi-machine scalability.

In [20]:
!pip install -q duckdb

In [21]:
!wget -O /content/duckdb "https://raw.githubusercontent.com/sjsu-cs131-f26/team4_medicaid_analysis/main/pa3/sql/analysis_text.sql"

--2026-11-12 03:19:55--  https://raw.githubusercontent.com/sjsu-cs131-f26/team4_medicaid_analysis/main/pa3/sql/analysis_text.sql
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2406 (2.3K) [text/plain]
Saving to: ‘/content/duckdb’

/content/duckdb     100%[===================>]   2.35K  --.-KB/s    in 0s      

2026-11-12 03:19:55 (34.3 MB/s) - ‘/content/duckdb’ saved [2406/2406]



In [22]:
import duckdb
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

input_path = "/content/analysis_ready.tsv"

quality_checks.toPandas().to_csv(input_path, sep="\t", index=False)

assert Path(input_path).is_file(), "Set input_path to your prepared TSV file."

sql = Path("/content/duckdb").read_text()

sql = sql.replace("../work/analysis_ready.tsv", input_path)
sql = sql.replace(
    "../out/duckdb_text_result.tsv",
    "/content/duckdb_text_result.tsv"
)

con = duckdb.connect()
con.execute(sql).fetchall()

duck_result = con.execute("""
    SELECT *
    FROM duckdb_result
    ORDER BY HCPCS_CODE
""").fetchdf()

display(duck_result)

,HCPCS_CODE,LOWEST_PAID_MONTH,HIGHEST_PAID_MONTH,LOWEST_DEMAND_MONTH,HIGHEST_DEMAND_MONTH
0,A0100,2020-03-01,2020-03-01,2020-03-01,2020-03-01
1,A0425,2023-07-01,2020-06-01,2020-06-01,2023-07-01
2,A0427,2019-05-01,2019-05-01,2019-05-01,2019-05-01
3,A0428,2021-03-01,2021-03-01,2021-03-01,2021-03-01
4,A0433,2024-07-01,2024-07-01,2024-07-01,2024-07-01
...,...,...,...,...,...
107,V2100,2023-10-01,2023-10-01,2023-10-01,2023-10-01
108,V2104,2021-10-01,2021-10-01,2021-10-01,2021-10-01
109,V2784,2023-05-01,2019-04-01,2023-05-01,2019-04-01
110,V5008,2021-12-01,2021-12-01,2021-12-01,2021-12-01


In [23]:
import pandas as pd

columns = [
    "HCPCS_CODE",
    "LOWEST_PAID_MONTH",
    "HIGHEST_PAID_MONTH",
    "LOWEST_DEMAND_MONTH",
    "HIGHEST_DEMAND_MONTH"
]


spark_result = (
    analysis_1
    .select(*columns)
    .orderBy("HCPCS_CODE")
    .toPandas()
)

pd.testing.assert_frame_equal(
    duck_result[columns].reset_index(drop=True),
    spark_result.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-6
)

print(f"Verified: all {len(duck_result)} result rows match.")

Verified: all 112 result rows match.


### **Task 4: Run-time Benchmarking (PySpark vs. DuckDB)**
We will measure the exact execution time (input reading + query execution) for both systems across 3 consecutive runs and write the output directly into `benchmark.csv`.

In [24]:
import time
import os

input_bytes = os.path.getsize(input_path)
input_rows = quality_checks.count()
output_rows = len(duck_result)

benchmark_results = []

# DuckDB 3 run benchmark
for run in range(1, 4):
    con = duckdb.connect()

    start_time = time.time()
    con.execute(sql)
    duck_df = con.execute("SELECT * FROM duckdb_result ORDER BY HCPCS_CODE").fetchdf()
    elapsed = time.time() - start_time

    benchmark_results.append({
        "implementation": "DuckDB",
        "environment": "Colab Local",
        "input_rows": input_rows,
        "input_bytes": input_bytes,
        "format": "TSV",
        "run": run,
        "elapsed_seconds": round(elapsed, 4),
        "output_rows": output_rows,
        "result_verified": "True"
    })

# PySpark 3 run benchmark
for run in range(1, 4):
    spark.catalog.clearCache()

    start_time = time.time()

    # Read input with inferSchema off to prevent timestamp parsing mismatches
    spark_input = spark.read.option("header", "true").option("delimiter", "\t").option("inferSchema", "false").csv(input_path)

    hcpcs_sum_temp = (
        spark_input
        .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
        .agg(
            F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES_SUM"),
            F.sum("TOTAL_PAID").alias("TOTAL_PAID_SUM")
        )
    )
    paid_val = F.col("TOTAL_PAID_SUM")
    lines_val = F.col("TOTAL_CLAIM_LINES_SUM")
    paid_per_line_temp = F.when(lines_val == 0, paid_val * F.lit(float("inf"))).otherwise(paid_val / lines_val)
    monthly_temp = hcpcs_sum_temp.withColumn("PAID_PER_CLAIM_LINE", F.round(paid_per_line_temp, 2))

    ranked_temp = (
        monthly_temp
        .withColumn("paid_asc_rank", month_rank("PAID_PER_CLAIM_LINE"))
        .withColumn("paid_desc_rank", month_rank("PAID_PER_CLAIM_LINE", descending=True))
        .withColumn("demand_asc_rank", month_rank("TOTAL_CLAIM_LINES_SUM"))
        .withColumn("demand_desc_rank", month_rank("TOTAL_CLAIM_LINES_SUM", descending=True))
    )

    analysis_1_temp = (
        ranked_temp
        .groupBy("HCPCS_CODE")
        .agg(
            F.max(F.when(F.col("paid_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_PAID_MONTH"),
            F.max(F.when(F.col("paid_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_PAID_MONTH"),
            F.max(F.when(F.col("demand_asc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("LOWEST_DEMAND_MONTH"),
            F.max(F.when(F.col("demand_desc_rank") == 1, F.col("CLAIM_FROM_MONTH"))).alias("HIGHEST_DEMAND_MONTH")
        )
        .orderBy(F.col("HCPCS_CODE").asc_nulls_last())
    )

    spark_df = analysis_1_temp.toPandas()
    elapsed = time.time() - start_time


    verified = "True" if spark_df[columns].equals(duck_result[columns]) else "False"

    benchmark_results.append({
        "implementation": "PySpark",
        "environment": f"Local Spark {spark.version}",
        "input_rows": input_rows,
        "input_bytes": input_bytes,
        "format": "TSV",
        "run": run,
        "elapsed_seconds": round(elapsed, 4),
        "output_rows": output_rows,
        "result_verified": verified
    })

benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)

,implementation,environment,input_rows,input_bytes,format,run,elapsed_seconds,output_rows,result_verified
0,DuckDB,Colab Local,234,16039,TSV,1,0.0998,112,True
1,DuckDB,Colab Local,234,16039,TSV,2,0.0458,112,True
2,DuckDB,Colab Local,234,16039,TSV,3,0.0406,112,True
3,PySpark,Local Spark 4.0.0,234,16039,TSV,1,2.7265,112,False
4,PySpark,Local Spark 4.0.0,234,16039,TSV,2,2.9216,112,False
5,PySpark,Local Spark 4.0.0,234,16039,TSV,3,2.2288,112,False


In [28]:
benchmark_df.to_csv(OUTPUT_URI + "benchmark.csv", index=False)
print("Saved benchmark.csv successfully.")


Saved benchmark.csv successfully.
